Problem: Best-Selling Item per Month

* Given a table called online_retail
* The goal is to identify the best-selling item for each month

Objective

* Determine the item with the highest total sales in each month

Calculation

* Total sales is calculated as:
    total_paid = unitprice * quantity

Conditions

* Exclude rows where quantity is negative (these represent returns)
* Exclude rows where the invoice number starts with ‘C’ (cancellations)

Output

* Month
* Item description
* Total amount paid

Summary

* For each month:
    * Calculate total sales per item
    * Select the item with the highest total_paid

table: online_retail
columns: 
    invoice_number - if starts with 'C' - meaning cancelled transaction
    description
    quantity
    unit_price
    invoice_date

Task  - Find best selling item for each month


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
data = [
("10001", "LUNCH BAG SPACEBOY DESIGN", 10, 7.426, "2023-01-10"),
("10002", "REGENCY CAKESTAND 3 TIER", 5, 7.65, "2023-02-15"),
("10003", "PAPER BUNTING WHITE LACE", 20, 5.1, "2023-03-12"),
("10004", "SPACEBOY LUNCH BOX", 6, 3.9, "2023-04-18"),
("10005", "PAPER BUNTING WHITE LACE", 10, 5.1, "2023-05-20"),
("C10006", "LUNCH BAG SPACEBOY DESIGN", -5, 7.426, "2023-01-25"),
("10007", "REGENCY CAKESTAND 3 TIER", -2, 7.65, "2023-02-28")
]
columns = ["invoice_number", "description", "quantity", "unit_price", "invoice_date"]

df = spark.createDataFrame(data, columns)

In [0]:
df.display()

invoice_number,description,quantity,unit_price,invoice_date
10001,LUNCH BAG SPACEBOY DESIGN,10,7.426,2023-01-10
10002,REGENCY CAKESTAND 3 TIER,5,7.65,2023-02-15
10003,PAPER BUNTING WHITE LACE,20,5.1,2023-03-12
10004,SPACEBOY LUNCH BOX,6,3.9,2023-04-18
10005,PAPER BUNTING WHITE LACE,10,5.1,2023-05-20
C10006,LUNCH BAG SPACEBOY DESIGN,-5,7.426,2023-01-25
10007,REGENCY CAKESTAND 3 TIER,-2,7.65,2023-02-28


In [0]:
#Filter valid transactions
from pyspark.sql.functions import col

df_clean = df.filter(
    (col("invoice_number").startswith("C") == False) &
    (col("quantity") > 0)
).display()

invoice_number,description,quantity,unit_price,invoice_date
10001,LUNCH BAG SPACEBOY DESIGN,10,7.426,2023-01-10
10002,REGENCY CAKESTAND 3 TIER,5,7.65,2023-02-15
10003,PAPER BUNTING WHITE LACE,20,5.1,2023-03-12
10004,SPACEBOY LUNCH BOX,6,3.9,2023-04-18
10005,PAPER BUNTING WHITE LACE,10,5.1,2023-05-20


In [0]:
#Convert date and extract months
from pyspark.sql.functions import month, col, to_date

df_date = df_clean.withColumn(
    "invoice_date",
    to_date(col("invoice_date"), "M/d/yyyy")
).withColumn(
    "month",
    month(col("invoice_date"), "M/d/yyyy")
)

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-5641837208874244>, line 4
      1 #Convert date and extract months
      2 from pyspark.sql.functions import month, col, to_date
----> 4 df_date = df_clean.withColumn(
      5     "invoice_date",
      6     to_date(col("invoice_date"), "M/d/yyyy")
      7 ).withColumn(
      8     "month",
      9     month(col("invoice_date"), "M/d/yyyy")
     10 )

AttributeError: 'NoneType' object has no attribute 'withColumn'

In [0]:
#2
df_filtered = df.filter(
    (~col("invoice_number").startswith("C")) &
    (col("quantity") > 0)
)

In [0]:
#Total_paid column added
df_with_total = df_filtered.withColumn(
    "total_paid",
    col("quantity") * col("unit_price")
)

In [0]:
df_with_total.display()

invoice_number,description,quantity,unit_price,invoice_date,total_paid
10001,LUNCH BAG SPACEBOY DESIGN,10,7.426,2023-01-10,74.26
10002,REGENCY CAKESTAND 3 TIER,5,7.65,2023-02-15,38.25
10003,PAPER BUNTING WHITE LACE,20,5.1,2023-03-12,102.0
10004,SPACEBOY LUNCH BOX,6,3.9,2023-04-18,23.4
10005,PAPER BUNTING WHITE LACE,10,5.1,2023-05-20,51.0


In [0]:
#Convert invoice_date to months only 
df_with_month = df_with_total.withColumn(
    "month",
    month(to_date("invoice_date"))
)

In [0]:
df_with_month.display()

invoice_number,description,quantity,unit_price,invoice_date,total_paid,month
10001,LUNCH BAG SPACEBOY DESIGN,10,7.426,2023-01-10,74.26,1
10002,REGENCY CAKESTAND 3 TIER,5,7.65,2023-02-15,38.25,2
10003,PAPER BUNTING WHITE LACE,20,5.1,2023-03-12,102.0,3
10004,SPACEBOY LUNCH BOX,6,3.9,2023-04-18,23.4,4
10005,PAPER BUNTING WHITE LACE,10,5.1,2023-05-20,51.0,5


In [0]:
# add values here

In [0]:
#Agg per item per month 

df_grouped = df_with_month.groupBy("month","description")\
    .sum("total_paid")\
    .withColumnRenamed("sum(total_paid)","total_paid")

In [0]:
df_grouped.display()

month,description,total_paid
1,LUNCH BAG SPACEBOY DESIGN,74.26
2,REGENCY CAKESTAND 3 TIER,38.25
3,PAPER BUNTING WHITE LACE,102.0
4,SPACEBOY LUNCH BOX,23.4
5,PAPER BUNTING WHITE LACE,51.0


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("month").orderBy(desc("total_paid"))

df_ranked = df_grouped.withColumn(
    "rank",
    row_number().over(window_spec)
)

In [0]:
df_ranked.display()

month,description,total_paid,rank
1,LUNCH BAG SPACEBOY DESIGN,74.26,1
2,REGENCY CAKESTAND 3 TIER,38.25,1
3,PAPER BUNTING WHITE LACE,102.0,1
4,SPACEBOY LUNCH BOX,23.4,1
5,PAPER BUNTING WHITE LACE,51.0,1


In [0]:
df_result = df_ranked.filter(col("rank") == 1)\
    .select("month","description","total_paid")

df_result.display()

month,description,total_paid
1,LUNCH BAG SPACEBOY DESIGN,74.26
2,REGENCY CAKESTAND 3 TIER,38.25
3,PAPER BUNTING WHITE LACE,102.0
4,SPACEBOY LUNCH BOX,23.4
5,PAPER BUNTING WHITE LACE,51.0
